# 04 — 直接模式：信号采集与 I/O

本笔记本介绍 CompactStat 直接模式下的扩展 I/O 功能：

- **高速波形** — 以 10 µs 到 20 ms 的速率采集最多 256 个电流/电位样本
- **外部 DAC/ADC 端口** — 输出模拟电压并读取外部传感器
- **数字 I/O** — 通过位掩码控制数字输出线并读取输入线
- **AC 信号控制** — 设置阻抗实验的频率和幅度
- **MUX 通道** — 切换多路复用器通道

### 前提条件
- 驱动已打开，设备已连接（参见 `02_device_and_instance_management.ipynb`）
- 电池已配置（参见 `03_direct_mode_basics.ipynb`）
- 错误处理参考：`01_getting_started.ipynb` — 第 4 节

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
Pyvium.connect_device()
Pyvium.set_connection_mode(1)  # EStat 4EL
Pyvium.set_current_range(2)    # 100 µA
Pyvium.set_filter(2)           # 10 kHz — 高速波形需要
print("就绪")

## 1. 高速电流波形

> ⚠️ **当前不可用。** `get_current_trace` 调用 DLL 函数 `IV_getcurrenttrace`，无论参数或电池状态如何，均会使宿主进程崩溃。根本原因正与 Ivium 支持团队联合排查。以下代码仅供参考，待问题解决后用于测试。

`get_current_trace(n_points, interval)` 采集一组电流样本。

- `n_points`：样本数量（最多 256）
- `interval`：样本间隔时间（秒），范围 10 µs 到 20 ms

返回以安培为单位的电流值列表。

> **HiSpeed 模式说明：** 当 `interval` 低于 2 ms 时，IviumSoft 会自动进入 **HiSpeed 模式** — 采集期间数据存储在恒电位仪的内部缓冲区中，采集完成后才传输到 PC。此模式下每次采集的最大缓冲区大小为 **8192 个点**（CV 为 32768）。在单次 HiSpeed 采集中尝试超过 8192 个点将被截断。间隔 ≥ 2 ms 时，数据实时流传输，无缓冲区限制。

In [ ]:
N_POINTS = 256
INTERVAL = 0.001  # 样本间隔 1 ms → 总采集 256 ms

Pyvium.set_potential(0.1)
Pyvium.set_cell_on()
time.sleep(0.05)  # 稳定

current_trace = Pyvium.get_current_trace(N_POINTS, INTERVAL)
print(f"采集了 {len(current_trace)} 个样本")
print(f"平均电流: {np.mean(current_trace):.3e} A")
print(f"标准差  : {np.std(current_trace):.3e} A")

times = [i * INTERVAL * 1000 for i in range(len(current_trace))]  # ms
currents_uA = [c * 1e6 for c in current_trace]  # µA

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times, currents_uA, lw=0.8)
ax.set_xlabel("时间 (ms)")
ax.set_ylabel("电流 (µA)")
ax.set_title(f"电流波形 — {N_POINTS} 点，间隔 {INTERVAL*1000:.1f} ms")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 高速电位波形

> ⚠️ **当前不可用。** 与第 1 节相同的问题 — `IV_getpotentialtrace` 会使宿主进程崩溃。仅供参考。

与电流波形 API 相同，但读取电位。适用于检查电流阶跃下的电池响应。

In [ ]:
potential_trace = Pyvium.get_potential_trace(N_POINTS, INTERVAL)

times_ms   = [i * INTERVAL * 1000 for i in range(len(potential_trace))]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_ms, potential_trace, 'r-', lw=0.8)
ax.set_xlabel("时间 (ms)")
ax.set_ylabel("电位 (V)")
ax.set_title("电位波形")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Pyvium.set_cell_off()

## 3. 对比电流和电位波形

> ⚠️ **当前不可用。** 两个波形函数均会使宿主进程崩溃。仅供参考。

在相同条件下同时采集两个信号（两次独立采集），获得完整图像。

In [ ]:
Pyvium.set_potential(0.2)
Pyvium.set_cell_on()
time.sleep(0.05)

I_trace = Pyvium.get_current_trace(128, 0.0005)   # 128 点，0.5 ms 间隔
E_trace = Pyvium.get_potential_trace(128, 0.0005)

Pyvium.set_cell_off()

t_ms = [i * 0.5 for i in range(128)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(t_ms, [c * 1e6 for c in I_trace], 'b-', lw=0.8)
ax1.set_ylabel("电流 (µA)")
ax1.grid(True, alpha=0.3)

ax2.plot(t_ms, E_trace, 'r-', lw=0.8)
ax2.set_xlabel("时间 (ms)")
ax2.set_ylabel("电位 (V)")
ax2.grid(True, alpha=0.3)

fig.suptitle("电流和电位波形")
plt.tight_layout()
plt.show()

## 4. 外部 DAC 输出

CompactStat 在外部端口上有两个外部 DAC 通道（0 和 1）。可用于向外部设备输出模拟控制信号（例如触发灯、控制阀、与其他仪器同步）。

输出范围：通常为 ±10 V（请查阅设备规格）。

In [ ]:
# 将 DAC 通道 0 设置为 1.5 V
Pyvium.set_dac(0, 1.5)
print("DAC 通道 0 设置为 1.5 V")

# 将 DAC 通道 1 设置为 -0.5 V
Pyvium.set_dac(1, -0.5)
print("DAC 通道 1 设置为 -0.5 V")

# 完成后将两通道重置为 0 V
Pyvium.set_dac(0, 0.0)
Pyvium.set_dac(1, 0.0)
print("DAC 通道已重置为 0 V")

## 5. 外部 ADC 输入

`get_adc(channel)` 读取 8 个外部 ADC 输入通道（0–7）之一的电压。适用于与电化学测量同步读取外部传感器（温度、pH、流量等）。

In [ ]:
for channel in range(4):  # 读取通道 0–3
    voltage = Pyvium.get_adc(channel)
    print(f"ADC 通道 {channel}: {voltage:.4f} V")

## 6. 数字输出

`set_digital_output(bitmask)` 控制数字输出线。整数中的每一位对应一条数字输出线。

```
bitmask = 0b00000101  →  第 0 和第 2 线为高电平，其他为低电平
```

In [ ]:
# 将数字输出第 0 线设为高电平
Pyvium.set_digital_output(0b00000001)
print(f"数字输出: 0b{0b00000001:08b}  （第 0 线为高电平）")
time.sleep(0.5)

# 将第 0 和第 2 线设为高电平
Pyvium.set_digital_output(0b00000101)
print(f"数字输出: 0b{0b00000101:08b}  （第 0 和第 2 线为高电平）")
time.sleep(0.5)

# 所有线设为低电平
Pyvium.set_digital_output(0b00000000)
print("数字输出: 所有线为低电平")

## 7. 数字输入

`get_digital_input()` 返回数字输入线当前状态的位掩码。

In [ ]:
digin = Pyvium.get_digital_input()
print(f"数字输入位掩码 : {digin} (0b{digin:08b})")

# 解码各条线
for line in range(8):
    state = "高电平" if digin & (1 << line) else "低电平"
    print(f"  第 {line} 线: {state}")

## 8. AC 信号控制（用于手动 EIS）

`set_ac_frequency()` 和 `set_ac_amplitude()` 配置施加到电池的 AC 扰动信号。在直接模式下，这可让你以固定频率手动施加 AC 信号 — 适用于单频阻抗测量或在运行完整 EIS 方法前验证设置。

自动化 EIS 扫描请改用带有 EIS 方法文件的 `start_method()`。

In [ ]:
# 设置 AC 扰动：1 kHz 下幅度 10 mV
Pyvium.set_ac_frequency(1000.0)   # Hz
Pyvium.set_ac_amplitude(0.010)    # V（10 mV）
print("AC 信号: 1 kHz，10 mV")

Pyvium.set_potential(0.0)         # DC 偏置
Pyvium.set_cell_on()
time.sleep(0.5)

# 读取产生的电流，粗略估算阻抗
I = Pyvium.get_current()
E = Pyvium.get_potential()
print(f"DC 电位: {E:.4f} V | DC 电流: {I:.3e} A")

Pyvium.set_cell_off()

## 9. MUX 通道选择

`set_mux_channel(n)` 将多路复用器切换到指定通道（从 0 开始编号）。当连接了多路复用器附件以轮流切换多个工作电极时使用。

In [ ]:
NUM_MUX_CHANNELS = 4
DWELL_TIME = 1.0  # 每通道停留秒数

Pyvium.set_potential(0.0)
Pyvium.set_cell_on()

results = []
for ch in range(NUM_MUX_CHANNELS):
    Pyvium.set_mux_channel(ch)
    time.sleep(DWELL_TIME)
    I = Pyvium.get_current()
    results.append((ch, I))
    print(f"  MUX 通道 {ch}: I = {I:.3e} A")

Pyvium.set_cell_off()
Pyvium.set_mux_channel(0)  # 重置到通道 0

## 清理

In [ ]:
Pyvium.set_cell_off()
Pyvium.disconnect_device()
Pyvium.close_driver()
print("完成")

---

## 小结

| 功能 | 方法 | 状态 | 主要限制 |
|-----------|--------|--------|----------------|
| 电流突发采集 | `get_current_trace(n, interval)` | ⚠️ 不可用 | DLL 崩溃 — 排查中 |
| 电位突发采集 | `get_potential_trace(n, interval)` | ⚠️ 不可用 | DLL 崩溃 — 排查中 |
| WE2 电流突发采集 | `get_current_we2_trace(n, interval)` | ⚠️ 不可用 | DLL 崩溃 — 排查中 |
| 模拟输出 | `set_dac(channel, volts)` | ✅ | 通道 0 或 1 |
| 模拟输入 | `get_adc(channel)` | ✅ | 通道 0–7 |
| 数字输出 | `set_digital_output(bitmask)` | ✅ | 整数位掩码 |
| 数字输入 | `get_digital_input()` | ✅ | 返回位掩码 |
| AC 频率 | `set_ac_frequency(hz)` | ✅ | 浮点数，单位 Hz |
| AC 幅度 | `set_ac_amplitude(volts)` | ✅ | 浮点数，单位 V |
| MUX 通道 | `set_mux_channel(n)` | ✅ | 从 0 开始；需要 MUX 附件 |

## 下一步

- **`05_bipotentiostat_and_we32.ipynb`** — BiStat WE2 通道和 32 通道 WE32 阵列
- **`06_method_mode.ipynb`** — 完整电化学方法执行（LSV、CV、EIS）